# 04 · Weather delay: extra trip time from bad weather

The app ranks truck routes by how much shelf life the strawberries have left on arrival, and that depends on **how long** the trip takes. Google's Routes API gives an arrival time (ETA) for each point along a route, but bad weather slows trucks down. This notebook turns the weather expected along a route into **extra minutes** beyond Google's estimate, using published speed reductions from the US Federal Highway Administration (FHWA).

**Reads:** nothing from disk. The speed reductions are typed in below, with their source.

**Produces:** a lookup table (speed reduction and time factor per weather bucket) and a function, `extra_delay_minutes`, that the backend logic will mirror.

*This is a lookup table, not a trained model. The project plan originally considered training a delay model on US Accidents data; v1 uses the FHWA table instead.*

## Step 1: the weather delay table

**What:** Stores how much each weather bucket reduces truck speed, and converts it into a **time factor**: how many times longer the same stretch takes.

**Why:** If speed drops by a fraction *r*, covering the same distance takes 1 / (1 − *r*) times as long. For example, a 20% slowdown makes a stretch take 1 / 0.8 = 1.25 times as long. The 8 buckets are the project's shared weather categories, so the backend's weather data can be mapped to them directly.

**Sources:**
- Rain, heavy rain and fog: midpoints of the speed-reduction ranges published by the FHWA Road Weather Management Program (https://ops.fhwa.dot.gov/weather/roadimpact.htm).
- Snow, ice and thunderstorm: **our assumptions**, labelled as such in the table.
- Clear and cloudy: no slowdown.

In [1]:
# Step 1: speed reduction per weather bucket, and the matching time factor
import pandas as pd

# Source: FHWA Road Weather Management Program, https://ops.fhwa.dot.gov/weather/roadimpact.htm
delay_table = pd.DataFrame(
    [
        ("clear", 0.00, "no slowdown"),
        ("cloudy", 0.00, "no slowdown"),
        ("rain", 0.08, "FHWA range midpoint"),
        ("heavy_rain", 0.10, "FHWA range midpoint"),
        ("fog", 0.11, "FHWA range midpoint"),
        ("snow", 0.20, "assumption"),
        ("ice", 0.30, "assumption"),
        ("thunderstorm", 0.10, "assumption"),
    ],
    columns=["bucket", "speed_reduction", "basis"],
).set_index("bucket")

# Speed drops by r -> the same distance takes 1 / (1 - r) times as long
delay_table["time_factor"] = 1 / (1 - delay_table["speed_reduction"])
print(delay_table.round(4).to_string())

              speed_reduction                basis  time_factor
bucket                                                         
clear                    0.00          no slowdown       1.0000
cloudy                   0.00          no slowdown       1.0000
rain                     0.08  FHWA range midpoint       1.0870
heavy_rain               0.10  FHWA range midpoint       1.1111
fog                      0.11  FHWA range midpoint       1.1236
snow                     0.20           assumption       1.2500
ice                      0.30           assumption       1.4286
thunderstorm             0.10           assumption       1.1111


## Step 2: extra minutes for a whole route

**What:** `extra_delay_minutes(route_points)` takes the route's points in driving order, each with an `eta` (expected arrival time) and a `weather_bucket`. For each stretch between two neighbouring points, it:
1. takes the planned minutes (difference between the two ETAs)
2. uses the **worse** weather of the stretch's two ends
3. adds planned minutes × (time factor − 1) extra minutes

It returns the total extra minutes for the route.

**Why:**
- We only know the weather at the points, not in between, so using the worse end is a cautious choice for a route that has to deliver on time.
- The function stops with a clear error if a bucket is unknown or the points are out of time order. Those values come from the backend, and a silent mistake there would give wrong delays.

In [2]:
# Step 2: extra minutes caused by weather along a route
SPEED_REDUCTION = delay_table["speed_reduction"].to_dict()
TIME_FACTOR = delay_table["time_factor"].to_dict()


def extra_delay_minutes(route_points):
    """Extra minutes beyond the given ETAs caused by weather, summed over the route.

    route_points: list of dicts in driving order, each with "eta" (a time) and "weather_bucket".
    Each stretch between two neighbouring points uses the worse bucket of its two ends.
    """
    for point in route_points:
        if point["weather_bucket"] not in TIME_FACTOR:
            raise ValueError(f"unknown weather bucket: {point['weather_bucket']!r}")

    total = 0.0
    for start, end in zip(route_points[:-1], route_points[1:]):
        planned_minutes = (pd.Timestamp(end["eta"]) - pd.Timestamp(start["eta"])) / pd.Timedelta(minutes=1)
        if planned_minutes < 0:
            raise ValueError("route points must be in ETA order")
        worse = max([start["weather_bucket"], end["weather_bucket"]], key=SPEED_REDUCTION.get)
        total += planned_minutes * (TIME_FACTOR[worse] - 1)
    return total

## Step 3: test on routes with known answers

**What:** Builds simple routes with a point every 10 minutes and checks the function against answers worked out by hand.

**Why:** If the function gets easy cases right, we can trust it on real routes. The `assert` lines stop the cell if any answer is wrong.
- 410 minutes of clear weather: no extra time.
- 410 minutes of rain: 410 × (1/0.92 − 1) ≈ 35.7 extra minutes.
- One 60-minute stretch from clear to snow uses the worse end (snow): 60 × (1/0.8 − 1) = 15 extra minutes.

In [3]:
# Step 3: tests with hand-calculated answers
def make_route(total_minutes, step_minutes, bucket):
    """Route points every step_minutes from 08:00, all in the same weather bucket."""
    start = pd.Timestamp("2026-07-01 08:00")
    return [
        {"eta": start + pd.Timedelta(minutes=m), "weather_bucket": bucket}
        for m in range(0, total_minutes + 1, step_minutes)
    ]


all_clear = extra_delay_minutes(make_route(410, 10, "clear"))
all_rain = extra_delay_minutes(make_route(410, 10, "rain"))
clear_to_snow = extra_delay_minutes([
    {"eta": "2026-07-01 08:00", "weather_bucket": "clear"},
    {"eta": "2026-07-01 09:00", "weather_bucket": "snow"},
])

print(f"410 min, all clear:  {all_clear:.1f} extra minutes")
print(f"410 min, all rain:   {all_rain:.1f} extra minutes")
print(f"60 min, clear->snow: {clear_to_snow:.1f} extra minutes")

assert all_clear == 0
assert round(all_rain, 1) == 35.7
assert abs(clear_to_snow - 15) < 1e-9
print("All tests passed")

410 min, all clear:  0.0 extra minutes
410 min, all rain:   35.7 extra minutes
60 min, clear->snow: 15.0 extra minutes
All tests passed


## Key findings

- **Weather is turned into time with one table:** speed reduction per bucket → time factor = 1 / (1 − reduction).
- **A 410-minute trip in rain takes about 36 minutes longer**, an 8.7% increase. Snow adds 25% and ice about 43% to the affected stretches.
- **Each stretch uses the worse weather of its two ends**, which is a cautious choice.

## Limitations

- **Snow, ice and thunderstorm values are assumptions**, not FHWA figures. Rain, heavy rain and fog use the midpoints of FHWA ranges, not exact values.
- **Speed only.** It ignores closures, accidents, and the extra stopping or chain-up time that severe weather can cause.
- **Weather is only known at route points.** Changes between points are invisible, and the "worse end" rule may overstate short weather patches.
- **Not trained or validated on real truck trips**, so the true delay for a given route is unknown.